In [ ]:
# 10_performance_benchmark.ipynb
# Performance benchmark: CSV loading + Autoencoder inference

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import time
import os

print("=== BENCHMARK WYDAJNOŚCI - INFERENCJA AUTOENCODERA ===")

# ========================================================
# 1. Wczytanie schematu cech
# ========================================================

normal_df = pd.read_csv("../data/processed/normal_features.csv")
feature_columns = normal_df.columns.tolist()
input_dim = len(feature_columns)

print(f"Liczba cech wejściowych: {input_dim}")

# ========================================================
# 2. Model definition consistent with 03_autoencoder.ipynb
# ========================================================

class ImprovedAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# ========================================================
# 3. Wczytanie wytrenowanego modelu
# ========================================================

model_path = "../models/best_autoencoder_final.pth"

model = ImprovedAutoencoder(input_dim)
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()

print(f"Model wczytany z: {model_path}")

# ========================================================
# 4. Lista scenariuszy
# ========================================================

scenarios = [
    "normal",
    "guloader",
    "scanning",
    "njrat",
    "kongtuke1",
    "kongtuke2",
    "remcos",
    "xloader",
    "xworm",
    "phantomstealer"
]

# ========================================================
# 5. Benchmark czystej inferencji
# ========================================================

results = []

for scenario in scenarios:
    path = f"../data/processed/{scenario}_features.csv"

    df = pd.read_csv(path)
    df = df.reindex(columns=feature_columns, fill_value=0)

    X = torch.tensor(df.values, dtype=torch.float32)

    # Rozgrzewka modelu, żeby pierwszy pomiar nie był zaburzony
    with torch.no_grad():
        _ = model(X)

    start = time.perf_counter()

    with torch.no_grad():
        recon = model(X)
        errors = torch.max((X - recon) ** 2, dim=1).values

    end = time.perf_counter()

    inference_time_s = end - start
    total_flows = len(df)

    flows_per_second = total_flows / inference_time_s if inference_time_s > 0 else 0
    time_per_flow_ms = (inference_time_s / total_flows) * 1000 if total_flows > 0 else 0

    results.append({
        "scenario": scenario,
        "total_flows": total_flows,
        "inference_time_s": round(inference_time_s, 6),
        "flows_per_second": round(flows_per_second, 2),
        "time_per_flow_ms": round(time_per_flow_ms, 6)
    })

df_benchmark = pd.DataFrame(results)

print("\n=== BENCHMARK CZYSTEJ INFERENCJI ===")
print(df_benchmark.to_string(index=False))

# ========================================================
# 6. Zapis wyników
# ========================================================

os.makedirs("../results", exist_ok=True)

output_path = "../results/performance_benchmark_inference.csv"
df_benchmark.to_csv(output_path, index=False)

print(f"\nSaved results to: {output_path}")

In [ ]:
# ========================================================
# 10_performance_benchmark.ipynb
# Performance benchmark: CSV loading + Autoencoder inference
# ========================================================

print("=== PERFORMANCE BENCHMARK - CSV LOADING + AUTOENCODER INFERENCE ===")

results_total = []

for scenario in scenarios:
    path = f"../data/processed/{scenario}_features.csv"

    start = time.perf_counter()

    df = pd.read_csv(path)
    df = df.reindex(columns=feature_columns, fill_value=0)
    X = torch.tensor(df.values, dtype=torch.float32)

    with torch.no_grad():
        recon = model(X)
        errors = torch.max((X - recon) ** 2, dim=1).values

    end = time.perf_counter()

    total_time_s = end - start
    total_flows = len(df)

    flows_per_second = total_flows / total_time_s if total_time_s > 0 else 0
    time_per_flow_ms = (total_time_s / total_flows) * 1000 if total_flows > 0 else 0

    results_total.append({
        "scenario": scenario,
        "total_flows": total_flows,
        "csv_load_plus_inference_time_s": round(total_time_s, 6),
        "flows_per_second": round(flows_per_second, 2),
        "time_per_flow_ms": round(time_per_flow_ms, 6)
    })

df_benchmark_total = pd.DataFrame(results_total)

print("\n=== CSV LOADING + INFERENCE RESULTS ===")
print(df_benchmark_total.to_string(index=False))

output_path = "../results/performance_benchmark_total.csv"
df_benchmark_total.to_csv(output_path, index=False)

print(f"\nSaved results to: {output_path}")